# ONNX-Tool Fusion and Validation

This notebook focuses on:
1. Conv + ReLU6(Clip) fusion with ONNX-Tool.
2. Output consistency validation in ONNX-Tool.


## Setup


In [1]:
import numpy as np
import onnx
from collections import Counter

from onnx_tool import loadmodel
from onnx_tool.fusion import FusionPattern


## Load Model


In [2]:
model_path = 'data/public/mobilenetv2-12/mobilenetv2-12.onnx'
model = loadmodel(model_path)
graph = model.graph
print('Model loaded:', model.modelname)
print('Graph nodes:', len(graph.nodemap))


Model loaded: mobilenetv2-12
Graph nodes: 105


## Build Compute Graph and Define Fusion Pattern


In [3]:
cg = graph.get_compute_graph()
print('Compute graph nodes (before fusion):', len(cg.nodemap))

ConvClip_pattern = [
    {
        'name': 'conv_0',
        'op': 'Conv',
        'attrs': [],
        'inport': [],
        'outport': [[0, 'clip_1', 0]],
    },
    {
        'name': 'clip_1',
        'op': 'Clip',
        'attrs': [],
        'inport': [[0, 'conv_0', 0]],
        'outport': [],
    },
]
pattern = FusionPattern(ConvClip_pattern)
found_nodes = pattern.search_pattern(cg)
print('Found Conv+Clip patterns:', len(found_nodes))


Compute graph nodes (before fusion): 100
Found Conv+Clip patterns: 35


## Perform Fusion in Compute Graph


In [4]:
for nodes in found_nodes:
    new_node_name = nodes[0].replace('conv', 'Conv_ReLU6')
    cg.fuse_subgraph_node_names(nodes, 'Conv_ReLU6', new_node_name, keep_attr=True, nodedomain='ai.custom')

cg.graph_reorder_nodes()

op_counts_after = Counter([n.op_type for n in cg.nodemap.values()])
print('Compute graph nodes (after fusion):', len(cg.nodemap))
print('Clip after fusion:', op_counts_after.get('Clip', 0))
print('Conv_ReLU6 after fusion:', op_counts_after.get('Conv_ReLU6', 0))


Compute graph nodes (after fusion): 65
Clip after fusion: 0
Conv_ReLU6 after fusion: 35


## Validate Consistency in ONNX-Tool


In [7]:
# Original compute graph (fresh)
original_model = loadmodel(model_path)
original_graph = original_model.graph.get_compute_graph()
original_graph.graph_reorder_nodes()

input_name = original_graph.input[0]
input_shape = original_graph.tensormap[input_name].shape
test_shape = [d if isinstance(d, int) and d > 0 else 1 for d in input_shape]
if len(test_shape) == 2:
    test_shape.extend([224, 224])
elif len(test_shape) == 3:
    test_shape.insert(1, 3)

shape_probe = np.zeros(test_shape, dtype=np.float32)
np.random.seed(42)
test_input = np.random.randn(*test_shape).astype(np.float32)

original_graph.shape_infer({input_name: shape_probe})
orig_out = original_graph.value_infer({input_name: test_input})[0]

fused_input_name = cg.input[0]
cg.shape_infer({fused_input_name: shape_probe})
fused_out = cg.value_infer({fused_input_name: test_input})[0]

print('allclose:', np.allclose(orig_out, fused_out, rtol=1e-5, atol=1e-5))
print('max abs diff:', float(np.max(np.abs(orig_out - fused_out))))


allclose: True
max abs diff: 0.0


## Export Fused Models

Export both variants:
- `mobilenetv2_fused.onnx`: from compute graph (`cg`), analysis-oriented.
- `mobilenetv2_fused_full.onnx`: from fresh full graph, deployment-oriented.


In [5]:
# Export compute-graph fused model (analysis-oriented)
output_path = 'data/public/mobilenetv2-12/mobilenetv2_fused.onnx'
cg.save_model(output_path, rawmodel=model.mproto)
print('Saved:', output_path)


Saved: data/public/mobilenetv2-12/mobilenetv2_fused.onnx


In [6]:
# Export full-graph fused model (ORT-ready)
model_for_export = loadmodel(model_path)
full_graph = model_for_export.graph
found_nodes_full = pattern.search_pattern(full_graph)

for nodes in found_nodes_full:
    new_node_name = nodes[0].replace('conv', 'Conv_ReLU6')
    full_graph.fuse_subgraph_node_names(
        nodes,
        'Conv_ReLU6',
        new_node_name,
        keep_attr=True,
        nodedomain='ai.custom',
    )

full_graph.graph_reorder_nodes()
output_path_ort = 'data/public/mobilenetv2-12/mobilenetv2_fused_full.onnx'
full_graph.save_model(output_path_ort, rawmodel=model_for_export.mproto, no_shape=True)
print('Saved:', output_path_ort)


Saved: data/public/mobilenetv2-12/mobilenetv2_fused_full.onnx


## Common Issues and Fixes

### 1) Incomplete exported model after `get_compute_graph()`
**Symptom**:
- Runtime load error like: node input is not graph input/initializer/previous output.
- Example: missing producer for shape tensor (such as `471`).

**Root cause**:
- `get_compute_graph()` removes shape-only branches for analysis.

**Fix**:
- Export deployment model from a fresh full graph:
  - `model_for_export = loadmodel(model_path)`
  - `full_graph = model_for_export.graph`
  - fuse on `full_graph`, then `save_model(...)`.


## QA

### Q1: Why can fused inference be slower in ONNX-Tool?
- ONNX-Tool `value_infer` is Python-heavy; Conv loops dominate.
- Fusion here is mainly graph transformation/correctness validation, not optimized kernel execution.

### Q2: Is `get_compute_graph()` wrong?
- Not wrong. It is designed for analysis/fusion/profiling workflows.
- It is not always directly suitable as deployment export graph.


## Summary

This notebook demonstrated:
1. Conv+Clip -> Conv_ReLU6 fusion in ONNX-Tool.
2. Output consistency validation inside ONNX-Tool.
3. Correct export guidance: use full graph for deployment.
